In [1]:
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

project_id = "dh-global-sales-data-dev"

Try to enrich manually labelled data with vendor_complete/vendor_history_complete

In [4]:
model_version_dict = {
    "AR": "5",
    "EG": "4"
}

In [6]:
def write_model_version_condition(country, version):
    model_version_condition = f"""('{country}', '{version}'),"""
    return model_version_condition

final_model_version_condition = ""

for key, value in model_version_dict.items():
    model_version_condition = write_model_version_condition(key, value)
    final_model_version_condition += model_version_condition
final_model_version_condition = final_model_version_condition.strip(",")

In [7]:
final_model_version_condition

"('AR', '5'),('EG', '4')"

In [12]:
#Extract data 
query = f"""
WITH labelled_data as(
    SELECT t1.*, `dh-global-sales-data.achilles.row_id_to_lead_id`(t1.left_row_id) AS left_lead_id, `dh-global-sales-data.achilles.row_id_to_lead_id`(t1.right_row_id) AS right_lead_id
    FROM `dh-global-sales-data-dev.achilles.training_candidates_new` AS t1
    WHERE (t1.country_iso, t1.model_version) IN ({final_model_version_condition})
    AND left_row_id not like 'ext_row_%'
    ORDER BY t1.country_iso, t1.model_version, t1.data_category_type
    )
SELECT labelled_data.*, vc1.restaurant_city as left_city, vc2.restaurant_city as right_city, vc1.main_cuisine as left_cuisine, vc2.main_cuisine as right_cuisine, vc1.price_level as left_price_level, vc2.price_level as right_price_level
FROM labelled_data
LEFT JOIN `dh-global-sales-data.leadgen_cl.vendor_complete` AS vc1 on labelled_data.left_lead_id = vc1.lead_id
LEFT JOIN `dh-global-sales-data.leadgen_cl.vendor_complete` AS vc2 on labelled_data.right_lead_id = vc2.lead_id
"""

df_training_candidates_new = pd.read_gbq(query=query, project_id = project_id, progress_bar_type = 'tqdm')

/opt/homebrew/anaconda3/envs/embedding_experimentation/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2309: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/homebrew/anaconda3/envs/embedding_experimentation/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2323: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/homebrew/anaconda3/envs/embedding_experimentation/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2337: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)


Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████|


No data in vendor complete for left leads. We'll try enriching with vendor_history_complete

In [27]:
df_training_candidates_new[df_training_candidates_new["left_city"].isna()==False]

,country_iso,left_row_id,left_name,left_street,left_lat,left_lng,left_phone_number,left_street_stop,left_name_stop,left_name_stop_phonetic,...,left_area,right_area,left_lead_id,right_lead_id,left_city,right_city,left_cuisine,right_cuisine,left_price_level,right_price_level


In [22]:
query = f"""
WITH labelled_data as(
    SELECT t1.*, `dh-global-sales-data.achilles.row_id_to_lead_id`(t1.left_row_id) AS left_lead_id, `dh-global-sales-data.achilles.row_id_to_lead_id`(t1.right_row_id) AS right_lead_id
    FROM `dh-global-sales-data-dev.achilles.training_candidates_new` AS t1
    WHERE (t1.country_iso, t1.model_version) IN ({final_model_version_condition})
    AND left_row_id not like 'ext_row_%'
    ORDER BY t1.country_iso, t1.model_version, t1.data_category_type
    )
SELECT labelled_data.*, vhc1.restaurant_city as left_city, vc2.restaurant_city as right_city, vhc1.main_cuisine as left_cuisine, vc2.main_cuisine as right_cuisine, vhc1.price_level as left_price_level, vc2.price_level as right_price_level
FROM labelled_data
LEFT JOIN `dh-global-sales-data-dev.leadgen_cl.vendor_history_complete` AS vhc1 on labelled_data.left_lead_id = vhc1.lead_id
LEFT JOIN `dh-global-sales-data.leadgen_cl.vendor_complete` AS vc2 on labelled_data.right_lead_id = vc2.lead_id
"""

df_training_candidates_new = pd.read_gbq(query=query, project_id = project_id, progress_bar_type = 'tqdm')

/opt/homebrew/anaconda3/envs/embedding_experimentation/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2309: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/homebrew/anaconda3/envs/embedding_experimentation/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2323: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/homebrew/anaconda3/envs/embedding_experimentation/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2337: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)


Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████|


No data in vendor history complete for left leads either. It seems we can't enrich manually labelled data, we'll have to work with only the columns in training_candidates_new

In [28]:
df_training_candidates_new[df_training_candidates_new["left_city"].isna()==False]

,country_iso,left_row_id,left_name,left_street,left_lat,left_lng,left_phone_number,left_street_stop,left_name_stop,left_name_stop_phonetic,...,left_area,right_area,left_lead_id,right_lead_id,left_city,right_city,left_cuisine,right_cuisine,left_price_level,right_price_level


In [46]:
df_training_candidates_new["left_country"]=""
df_training_candidates_new.loc[df_training_candidates_new["country_iso"]=="AR", "left_country"]="Argentina"
df_training_candidates_new.loc[df_training_candidates_new["country_iso"]=="EG","left_country"]="Egypt"
df_training_candidates_new["right_country"]=df_training_candidates_new["left_country"]

In [47]:
def generate_vendor_document(name, name_local, street, country):
    return f"""name: {name}, name_local: {name_local}, street: {street}, country: {country}"""

df_training_candidates_new["left_document"] = df_training_candidates_new.apply(lambda x: generate_vendor_document(x.left_name, x.left_name_local, x.left_street, x.left_country), axis=1)
df_training_candidates_new["right_document"] = df_training_candidates_new.apply(lambda x: generate_vendor_document(x.right_name, x.right_name_local, x.right_street, x.right_country), axis=1)

In [55]:
training_data = df_training_candidates_new[["left_document", "right_document", "label", "left_country"]]

## 1. Simple bi-ecoder for semantic matching based on text vendor data

In [51]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer

# Multilingual Bi-Encoder with a deeper projection network
class MultilingualBiEncoder(nn.Module):
    def __init__(self, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", hidden_size=256, num_layers=3):
        super(MultilingualBiEncoder, self).__init__()
        self.encoder = SentenceTransformer(model_name)
        self.transformer = self.encoder._first_module().auto_model  # Get actual Transformer model
        self.tokenizer = self.encoder.tokenizer  # Get tokenizer

        transformer_hidden_size = self.transformer.config.hidden_size
        
        # Deeper feed-forward neural network as projection layer
        layers = []
        input_size = transformer_hidden_size
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(input_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.1))  # Dropout for regularization
            input_size = hidden_size
        layers.append(nn.Linear(input_size, transformer_hidden_size))  # Output layer
        
        self.projection = nn.Sequential(*layers)

    def encode(self, text, device):
        """Encodes text into a trainable embedding."""
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {key: value.to(device) for key, value in inputs.items()}  # Move to GPU if available
        outputs = self.transformer(**inputs)
        embeddings = outputs.last_hidden_state[:, 0, :]  # Use CLS token embedding
        return self.projection(embeddings)  # Apply projection layer

    def forward(self, left_vendor_text, right_vendor_text, device):
        """Computes embeddings for both vendor descriptions."""
        left_vendor_embed = self.encode(left_vendor_text, device)
        right_vendor_embed = self.encode(right_vendor_text, device)
        return left_vendor_embed, right_vendor_embed

# Contrastive Loss
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, embed1, embed2, label):
        """Computes contrastive loss with cosine similarity."""
        similarity = F.cosine_similarity(embed1, embed2)
        loss = (1 - label) * (1 - similarity).pow(2) + label * F.relu(similarity - self.margin).pow(2)
        return loss.mean()

# Torch Dataset
class VendorDataset(Dataset):
    def __init__(self, dataframe):
        self.left_vendor = dataframe["left_document"].tolist()
        self.right_vendor = dataframe["right_document"].tolist()
        self.labels = torch.tensor(dataframe["label"].tolist(), dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.left_vendor[idx], self.right_vendor[idx], self.labels[idx]

# Training Function
def train(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for left_vendor, right_vendor, labels in dataloader:
        left_vendor_embed, right_vendor_embed = model(left_vendor, right_vendor, device)
        labels = labels.to(device)

        loss = criterion(left_vendor_embed, right_vendor_embed, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer

# Multilingual Bi-Encoder
class MultilingualBiEncoder(nn.Module):
    def __init__(self, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        super(MultilingualBiEncoder, self).__init__()
        self.encoder = SentenceTransformer(model_name)
        self.transformer = self.encoder._first_module().auto_model  # Get actual Transformer model
        self.tokenizer = self.encoder.tokenizer  # Get tokenizer

        hidden_size = self.transformer.config.hidden_size
        self.projection = nn.Linear(hidden_size, hidden_size)  # Optional projection layer

    def encode(self, text):
        """Encodes text into a trainable embedding."""
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {key: value.to(device) for key, value in inputs.items()}  # Move to GPU if available
        outputs = self.transformer(**inputs)
        embeddings = outputs.last_hidden_state[:, 0, :]  # Use CLS token embedding
        return self.projection(embeddings)  # Optional projection


    def forward(self, left_vendor_text, right_vendor_text):
        """Computes embeddings for both vendor descriptions."""
        left_vendor_embed = self.encode(left_vendor_text)
        right_vendor_embed = self.encode(right_vendor_text)
        return left_vendor_embed, right_vendor_embed

# Contrastive Loss
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, embed1, embed2, label):
        """Computes contrastive loss with cosine similarity."""
        similarity = F.cosine_similarity(embed1, embed2)
        loss = (1 - label) * (1 - similarity).pow(2) + label * F.relu(similarity - self.margin).pow(2)
        return loss.mean()

# Torch Dataset
class VendorDataset(Dataset):
    def __init__(self, dataframe):
        self.left_vendor = dataframe["left_document"].tolist()
        self.right_vendor = dataframe["right_document"].tolist()
        self.labels = torch.tensor(dataframe["label"].tolist(), dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.left_vendor[idx], self.right_vendor[idx], self.labels[idx]

# Training Function
def train(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for left_vendor, right_vendor, labels in dataloader:
        left_vendor_embed, right_vendor_embed = model(left_vendor, right_vendor)
        labels = labels.to(device)

        loss = criterion(left_vendor_embed, right_vendor_embed, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [53]:
dataset = VendorDataset(training_data)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# Initialize Model, Loss, and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultilingualBiEncoder().to(device)
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
print(f"Model initialized")

# Train the Model
num_epochs = 5
print(f"Training starting")
for epoch in range(num_epochs):
    loss = train(model, dataloader, criterion, optimizer, device)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss:.4f}")


README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

Model initialized
Training starting
Epoch 1/5, Loss: 0.0003
Epoch 2/5, Loss: 0.0001
Epoch 3/5, Loss: 0.0000
Epoch 4/5, Loss: 0.0000
Epoch 5/5, Loss: 0.0000


In [58]:
training_data

,left_document,right_document,label,left_country
0,"name: remy, name_local: remy, street: khalid b...","name: remy - sidi bishr, name_local: remy - si...",1,Egypt
1,"name: la tana minimercado, name_local: la tana...","name: duplicado la tana, name_local: duplicado...",1,Argentina
2,"name: beer market, name_local: beer market, st...","name: almacén de bebidas hans beer, name_local...",0,Argentina
3,"name: walmart santa fe, name_local: walmart sa...","name: mostaza sta fe walmart ribera, name_loca...",1,Argentina
4,"name: necochea, name_local: necochea, street: ...","name: la juanita necochea, name_local: la juan...",1,Argentina
...,...,...,...,...
5172,"name: grido barrio norte, name_local: grido ba...","name: grido, name_local: grido, street: av col...",0,Argentina
5173,"name: florizza, name_local: florizza, street: ...","name: florizza, name_local: florizza, street: ...",1,Argentina
5174,"name: alessandro helados caballito, name_local...","name: caballito, name_local: caballito, street...",0,Argentina
5175,"name: import coffee company retiro, name_local...","name: import coffee company, name_local: impor...",1,Argentina


In [62]:
#Example Inference
left_vendor_text = training_data["left_document"][2]
right_vendor_text = training_data["right_document"][2]

embed1, embed2 = model(left_vendor_text, right_vendor_text, "cpu")
similarity = F.cosine_similarity(embed1, embed2)
print(f"Similarity Score: {similarity.item():.4f}")

Similarity Score: 0.9971


In [63]:
left_vendor_text

'name: beer market, name_local: beer market, street: tinogasta, country: Argentina'

In [64]:
right_vendor_text

'name: almacén de bebidas hans beer, name_local: almacén de bebidas hans beer, street: tinogasta, country: Argentina'

In [65]:
#Example Inference
left_vendor_text = training_data["left_document"][1]
right_vendor_text = training_data["right_document"][1]

embed1, embed2 = model(left_vendor_text, right_vendor_text, "cpu")
similarity = F.cosine_similarity(embed1, embed2)
print(f"Similarity Score: {similarity.item():.4f}")

Similarity Score: 0.9928


In [67]:
left_vendor_text

'name: la tana minimercado, name_local: la tana minimercado, street: cordoba, country: Argentina'

In [66]:
right_vendor_text

'name: duplicado la tana, name_local: duplicado la tana, street: cordoba 575, country: Argentina'

In [68]:
#Example Inference
left_vendor_text = training_data["left_document"][0]
right_vendor_text = training_data["right_document"][0]

embed1, embed2 = model(left_vendor_text, right_vendor_text, "cpu")
similarity = F.cosine_similarity(embed1, embed2)
print(f"Similarity Score: {similarity.item():.4f}")

Similarity Score: 0.9966


In [70]:
left_vendor_text

'name: remy, name_local: remy, street: khalid bin al waleed street 42, country: Egypt'

In [71]:
right_vendor_text

'name: remy - sidi bishr, name_local: remy - sidi bishr, street: 41 bashir al- shendi street formerly health off khaled ibn al- waleed sidi bishr tram in front of the new abdullah pharmacy sidi bishr, country: Egypt'

In [72]:
#Example Inference
left_vendor_text = training_data["left_document"][0]
right_vendor_text = training_data["right_document"][1]

embed1, embed2 = model(left_vendor_text, right_vendor_text, "cpu")
similarity = F.cosine_similarity(embed1, embed2)
print(f"Similarity Score: {similarity.item():.4f}")

Similarity Score: 0.9988


In [73]:
left_vendor_text

'name: remy, name_local: remy, street: khalid bin al waleed street 42, country: Egypt'

In [74]:
right_vendor_text

'name: duplicado la tana, name_local: duplicado la tana, street: cordoba 575, country: Argentina'